# Technical-indicator feature ablation

This notebook compares the existing base predictors with the same predictors plus MACD, RSI, Bollinger bandwidth and session VWAP distance. Both feature sets use identical eligible rows, five chronological folds, model settings and out-of-sample timestamps. The saved tables are loaded here; model training ran in an isolated process. Indicators use current and earlier bars only. The session VWAP uses the available IEX volume in these files.

This is an analysis notebook, not dissertation prose. The results are exploratory and do not establish that the indicators improve future trading performance.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
results_dir = project_root / 'results'
fold_metrics = pd.read_csv(results_dir / 'indicator_ablation_fold_metrics.csv')
predictions = pd.read_csv(results_dir / 'indicator_ablation_predictions.csv', parse_dates=['datetime', 'next_datetime'])
summary = pd.read_csv(results_dir / 'indicator_ablation_summary.csv')
deltas = pd.read_csv(results_dir / 'indicator_ablation_deltas.csv')
backtest = pd.read_csv(results_dir / 'indicator_ablation_backtest_summary.csv')

## Mean fold metrics

In [ ]:
display(summary.sort_values(['ETF', 'model', 'feature_set']).round(4))
display(deltas.sort_values(['ETF', 'model']).round(4))

## Cost-sensitive XGBoost backtest

In [ ]:
xgb_backtest = backtest.loc[backtest['model'].eq('XGBoost')].copy()
display(xgb_backtest.sort_values(['ETF', 'feature_set', 'cost_bps_per_side']).round(4))

## Coverage and timestamp alignment

The number of out-of-sample predictions and unique timestamps should match across the two feature sets for each ETF.

In [ ]:
coverage = (predictions.groupby(['ETF', 'feature_set'])
            .agg(rows=('datetime', 'size'), unique_timestamps=('datetime', 'nunique'))
            .reset_index())
display(coverage)
assert predictions.groupby('ETF')['feature_set'].nunique().eq(2).all()
assert not predictions.duplicated(['ETF', 'feature_set', 'datetime']).any()